# DKIST Observatory Overview — Implementation / DKIST 관측소 개요 — 구현

**Paper**: Rimmele, T. R., Warner, M., Keil, S. L., et al., "The Daniel K. Inouye Solar Telescope – Observatory Overview", *Solar Physics*, Vol. 295, Article 172 (2020). [DOI: 10.1007/s11207-020-01736-7]

This notebook reproduces the core quantitative arguments of the DKIST overview paper through numerical calculation and visualization.

이 노트북은 DKIST overview 논문의 핵심 정량적 논증을 수치 계산과 시각화로 재현합니다.

## Topics Covered / 다루는 주제

1. **Diffraction-limited resolution / 회절 한계 분해능** — why 4 m is the minimum aperture
2. **Zeeman splitting and IR advantage / 제이만 분열과 IR 이점** — $\lambda^2$ scaling
3. **AO error budget and Strehl ratio / AO 오차 예산과 Strehl 비** — fitting + bandwidth errors
4. **Collecting area and coronal signal / 집광 면적과 코로나 신호** — DKIST vs. Solar-C
5. **Heat load and flux at prime focus / 열 부하와 prime focus 플럭스** — 2.5 MW/m²
6. **Polarimetric photon budget / 편광 광자 예산** — achieving $5 \times 10^{-4}$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 100

# Physical constants
C_LIGHT = 2.998e8         # m/s
E_CHARGE = 1.602e-19      # C
M_ELECTRON = 9.109e-31    # kg
AU = 1.496e11             # m
SOLAR_FLUX = 1361.0       # W/m^2 (TSI at 1 AU)
ARCSEC_TO_RAD = np.pi / (180.0 * 3600.0)

## Part 1: Diffraction-Limited Resolution / 회절 한계 분해능

The Airy diffraction limit for a circular aperture:

$$
\theta_{\text{diff}} = 1.22 \, \frac{\lambda}{D}
$$

DKIST의 4 m 구경은 태양 대기의 **광자 평균자유행로 (photon mean-free path)**와 **압력 스케일 높이 (pressure scale height)** 수준인 ~70 km 이하를 분해하기 위해 선택되었다. Stenflo (2008)는 이를 태양 대기의 "critical scale"이라 정의했다.

DKIST's 4 m aperture was chosen to resolve scales at or below the photon mean-free path and pressure scale height (~70 km) of the solar atmosphere — what Stenflo (2008) calls the "critical scale".

In [ ]:
def diffraction_limit(wavelength_nm, aperture_m):
    """Compute Airy diffraction-limited angular resolution.

    Args:
        wavelength_nm: Observing wavelength in nanometers.
        aperture_m: Telescope aperture diameter in meters.

    Returns:
        tuple (theta_arcsec, linear_km): Angular resolution in arcsec and
            linear scale on the Sun in km.
    """
    wavelength_m = wavelength_nm * 1e-9
    theta_rad = 1.22 * wavelength_m / aperture_m
    theta_arcsec = theta_rad / ARCSEC_TO_RAD
    linear_km = theta_rad * AU * 1e-3
    return theta_arcsec, linear_km


# Compare telescopes at 500 nm and 1600 nm
apertures = {
    'DST (Sacramento Peak)': 0.76,
    'SST (La Palma)': 1.0,
    'GREGOR (Tenerife)': 1.5,
    'GST (BBSO)': 1.6,
    'DKIST (Haleakala)': 4.0,
    'EST (future)': 4.0,
    'CGST (planned)': 8.0,
}

print(f"{'Telescope':<25s} {'D [m]':>7s} {'500 nm':>18s} {'1600 nm':>18s}")
print('-' * 72)
for name, D in apertures.items():
    theta_vis, km_vis = diffraction_limit(500, D)
    theta_ir, km_ir = diffraction_limit(1600, D)
    print(f"{name:<25s} {D:>7.2f} {theta_vis:>8.3f}\" ({km_vis:>4.0f} km) {theta_ir:>8.3f}\" ({km_ir:>4.0f} km)")

In [ ]:
# Visualize diffraction limit vs aperture at multiple wavelengths
D_range = np.linspace(0.5, 8.0, 100)
wavelengths = [500, 630, 1083, 1600, 3935, 5000]  # nm
wavelength_labels = [
    '500 nm (Fe I lines)',
    '630 nm (Fe I)',
    '1083 nm (He I)',
    '1600 nm (opacity min)',
    '3935 nm (Si IX coronal)',
    '5000 nm (mid-IR)',
]

fig, ax = plt.subplots(figsize=(11, 7))
for lam, label in zip(wavelengths, wavelength_labels):
    _, km = np.vectorize(diffraction_limit)(lam, D_range)
    ax.plot(D_range, km, label=label, linewidth=2)

ax.axhline(70, color='red', linestyle='--', alpha=0.6, label='Pressure scale height (70 km)')
ax.axhline(20, color='darkred', linestyle=':', alpha=0.6, label='Photon mean-free path (~20 km)')
ax.axvline(4.0, color='black', linestyle='-.', alpha=0.5, label='DKIST (4 m)')

ax.set_xlabel('Aperture D [m]')
ax.set_ylabel('Linear resolution on Sun [km]')
ax.set_title('Diffraction-limited linear resolution vs. aperture')
ax.set_xlim(0.5, 8.0)
ax.set_ylim(5, 500)
ax.set_yscale('log')
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

print('\nKey insight: Only D >= 4 m can resolve the photon mean-free path at 1.6 microns,')
print('which is exactly why DKIST was sized at 4 m.')

## Part 2: Zeeman Splitting — Why IR? / 제이만 분열 — 왜 IR인가?

Normal Zeeman effect:

$$
\Delta\lambda_B = \frac{e}{4\pi m_e c} \, g_{\text{eff}} \, \lambda^2 \, B
$$

The $\lambda^2$ scaling means IR lines exhibit much larger Zeeman splitting than visible lines — this is a *fundamental* advantage of IR spectro-polarimetry for weak magnetic field detection.

$\lambda^2$ 스케일링으로 IR 라인은 가시광 대비 훨씬 큰 제이만 분열을 보인다 — 이것이 **약한 자기장 검출에서 IR 분광편광측정의 근본적 이점**이다.

In [ ]:
def zeeman_splitting_mA(wavelength_nm, B_gauss, g_eff):
    """Compute normal Zeeman wavelength splitting in milliangstroms.

    Args:
        wavelength_nm: Rest wavelength of spectral line in nm.
        B_gauss: Magnetic field strength in Gauss.
        g_eff: Effective Lande factor.

    Returns:
        Zeeman wavelength shift in milliangstroms (mA).
    """
    # Convert to SI
    wavelength_m = wavelength_nm * 1e-9
    B_tesla = B_gauss * 1e-4
    # e / (4 pi m_e c) in SI (units: 1/(T*m) when multiplied by lambda^2 * B)
    coeff = E_CHARGE / (4.0 * np.pi * M_ELECTRON * C_LIGHT)
    dlambda_m = coeff * g_eff * wavelength_m**2 * B_tesla
    # Convert to milliangstrom (1 A = 1e-10 m, 1 mA = 1e-13 m)
    return dlambda_m * 1e13


# Key spectral lines used in solar magnetometry
lines = [
    (525.02, 3.0, 'Fe I 525.0 (vis)'),
    (630.15, 1.67, 'Fe I 630.15 (vis)'),
    (630.25, 2.5,  'Fe I 630.25 (vis)'),
    (1083.0, 1.0,  'He I 10830 (chromo)'),
    (1564.85, 3.0, 'Fe I 15648 (IR)'),
    (3935.0, 2.0,  'Si IX 3935 (coronal)'),
    (12300.0, 1.5, 'Mg I 12318 (photosphere)'),
]

B_test = 100.0  # Gauss
print(f"Zeeman splitting at B = {B_test} G:\n")
print(f"{'Line':<28s} {'lambda [nm]':>12s} {'g':>6s} {'dLambda [mA]':>14s}")
print('-' * 62)
for lam, g, label in lines:
    dlam = zeeman_splitting_mA(lam, B_test, g)
    print(f"{label:<28s} {lam:>12.2f} {g:>6.2f} {dlam:>14.2f}")

In [ ]:
# Plot Zeeman splitting vs wavelength for a fixed B
lambda_range = np.linspace(300, 5000, 500)  # nm
B_values = [10, 100, 1000]  # Gauss

fig, ax = plt.subplots(figsize=(11, 6))
for B in B_values:
    # Assume unit Lande factor for universal curve; multiply by g for specific lines
    dlam = zeeman_splitting_mA(lambda_range, B, g_eff=1.0)
    ax.plot(lambda_range, dlam, label=f'B = {B} G (g=1)', linewidth=2)

# Highlight DKIST-relevant lines
for lam, g, label in lines:
    if lam < 5000:
        dlam = zeeman_splitting_mA(lam, 100, g)
        ax.scatter([lam], [dlam], s=60, zorder=5)
        ax.annotate(label, (lam, dlam), fontsize=8,
                    textcoords='offset points', xytext=(5, 5))

ax.set_xlabel('Wavelength [nm]')
ax.set_ylabel('Zeeman splitting dLambda [mA]')
ax.set_title('Zeeman splitting grows as lambda^2 — fundamental IR advantage')
ax.set_yscale('log')
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper left')
ax.set_xlim(300, 5000)
plt.tight_layout()
plt.show()

ratio = zeeman_splitting_mA(1564.85, 100, 3.0) / zeeman_splitting_mA(630.25, 100, 2.5)
print(f'\nFe I 1564.85 nm vs Fe I 630.25 nm splitting ratio at 100 G: {ratio:.1f} times')
print('-> IR line is ~7-8x more sensitive to the same magnetic field')

## Part 3: AO Error Budget and Strehl Ratio / AO 오차 예산과 Strehl 비

The Strehl ratio after AO correction:

$$
S = \exp(-\sigma_\phi^2), \qquad \sigma_\phi^2 = \sigma_{\text{fit}}^2 + \sigma_{\text{BW}}^2 + \sigma_{\text{noise}}^2 + \sigma_{\text{aniso}}^2
$$

Fitting and bandwidth errors:

$$
\sigma_{\text{fit}}^2 = 0.28 \left(\frac{d}{r_0}\right)^{5/3}, \qquad \sigma_{\text{BW}}^2 = \left(\frac{\tau}{\tau_0}\right)^{5/3}
$$

DKIST: $d \approx 10$ cm (1,457 sub-apertures across 4 m), $\tau = 0.5$ ms (2 kHz).

DKIST의 Strehl 요구사항: $S(500\,\text{nm}) > 0.3$ at $r_0 > 7$ cm; $S(630\,\text{nm}) > 0.6$ at $r_0 > 20$ cm.

In [ ]:
def strehl_ratio(r0_m, wavelength_nm, d_subap_m=0.10, tau_ms=0.5, v_wind_ms=10.0):
    """Compute DKIST AO Strehl ratio from fitting and bandwidth errors.

    r0 is specified at 500 nm; rescales to target wavelength via r0 ~ lambda^(6/5).

    Args:
        r0_m: Fried parameter at 500 nm in meters.
        wavelength_nm: Target wavelength in nm.
        d_subap_m: Sub-aperture (and actuator) pitch in meters.
        tau_ms: AO update period in milliseconds.
        v_wind_ms: Wind-weighted atmospheric speed in m/s.

    Returns:
        tuple (S, sigma_fit_sq, sigma_bw_sq): Strehl and variance components.
    """
    # Rescale r0 to target wavelength
    r0_target = r0_m * (wavelength_nm / 500.0)**(6.0 / 5.0)

    # Greenwood time (s)
    tau0 = 0.314 * r0_target / v_wind_ms
    tau = tau_ms * 1e-3

    sigma_fit_sq = 0.28 * (d_subap_m / r0_target)**(5.0 / 3.0)
    sigma_bw_sq = (tau / tau0)**(5.0 / 3.0)

    S = np.exp(-(sigma_fit_sq + sigma_bw_sq))
    return S, sigma_fit_sq, sigma_bw_sq


# Validate DKIST requirements
print('DKIST Strehl requirements check:')
print('-' * 60)
for r0_500, lam, S_req in [(0.07, 500, 0.3), (0.20, 630, 0.6)]:
    S, fit, bw = strehl_ratio(r0_500, lam)
    status = 'PASS' if S >= S_req else 'FAIL'
    print(f'r0(500)={r0_500*100:.0f} cm, lam={lam} nm: '
          f'S={S:.3f} (req {S_req:.2f}) [{status}]')
    print(f'  fitting sigma^2 = {fit:.4f}, bandwidth sigma^2 = {bw:.4f}')

In [ ]:
# Plot Strehl vs r0 at multiple wavelengths
r0_range = np.linspace(0.03, 0.30, 100)
wavelengths_plot = [400, 500, 630, 854, 1083, 1600]

fig, ax = plt.subplots(figsize=(11, 7))
for lam in wavelengths_plot:
    S = np.array([strehl_ratio(r0, lam)[0] for r0 in r0_range])
    ax.plot(r0_range * 100, S, label=f'{lam} nm', linewidth=2)

# DKIST requirements
ax.scatter([7], [0.3], s=100, marker='s', color='black', zorder=5,
           label='DKIST req: S(500)>=0.3 @ r0=7 cm')
ax.scatter([20], [0.6], s=100, marker='^', color='red', zorder=5,
           label='DKIST req: S(630)>=0.6 @ r0=20 cm')

ax.set_xlabel('r0 at 500 nm [cm]')
ax.set_ylabel('Strehl ratio S')
ax.set_title('DKIST AO Strehl ratio vs. seeing conditions')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right', fontsize=9)
ax.set_xlim(3, 30)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print('\nKey insight: Longer wavelengths achieve much higher Strehl for the same seeing.')
print('This is why DKIST IR instruments (DL-NIRSP, CRYO-NIRSP) routinely reach S > 0.9.')

## Part 4: Collecting Area and Coronal Signal / 집광 면적과 코로나 신호

Coronal observations are inherently *photon-starved*:

$$
\frac{I_{\text{corona}}}{I_{\text{disk}}} \sim 10^{-6}, \qquad \frac{\delta I_{\text{pol}}}{I_{\text{corona}}} \sim 10^{-3}\text{–}10^{-4}
$$

The polarimetric signal from the coronal magnetic field is $\sim 10^{-9}$ relative to the disk. DKIST's 4 m aperture delivers 75× the collecting area of the 46 cm Solar-C coronagraph.

코로나 관측은 본질적으로 **photon-starved** — 코로나 자기장 편광 신호는 디스크 대비 $\sim 10^{-9}$ 수준이다. DKIST 4 m 구경은 46 cm Solar-C 대비 **75배의 집광 면적**을 제공한다.

In [ ]:
def collecting_area_ratio(D1_m, D2_m):
    """Ratio of collecting areas between two circular apertures."""
    return (D1_m / D2_m)**2


def integration_time_for_snr(I_signal_rel, snr_target, photon_rate_hz_per_m2, D_m):
    """Estimate integration time to achieve target SNR for a faint signal.

    Photon-limited: SNR = sqrt(N_photons) for strong continuum, but
    for a faint signal on top of a larger continuum the relevant shot
    noise comes from the total photon count N and SNR = I_signal * sqrt(N).

    Args:
        I_signal_rel: Signal fractional intensity (e.g. 1e-4).
        snr_target: Desired SNR on the signal.
        photon_rate_hz_per_m2: Photon rate per unit collecting area (Hz/m^2).
        D_m: Telescope diameter.

    Returns:
        Integration time in seconds.
    """
    A = np.pi * (D_m / 2.0)**2
    rate = photon_rate_hz_per_m2 * A
    N_required = (snr_target / I_signal_rel)**2
    return N_required / rate


# Compare DKIST vs Solar-C for coronal magnetometry
D_dkist = 4.0
D_solarc = 0.46
area_ratio = collecting_area_ratio(D_dkist, D_solarc)
print(f'Collecting area ratio DKIST / Solar-C: {area_ratio:.1f} x')
print(f'(DKIST area: {np.pi * 2.0**2:.2f} m^2, Solar-C: {np.pi * 0.23**2:.3f} m^2)\n')

# Assume a baseline photon rate (illustrative)
photon_rate = 1e12  # photons/s/m^2 (bright coronal emission line, illustrative)
signal_frac = 1e-4  # polarimetric signal relative to coronal intensity
snr = 10.0

t_dkist = integration_time_for_snr(signal_frac, snr, photon_rate, D_dkist)
t_solarc = integration_time_for_snr(signal_frac, snr, photon_rate, D_solarc)
print(f'Integration time for SNR=10 at signal=1e-4 (illustrative photon rate):')
print(f'  DKIST:   {t_dkist:.1f} s')
print(f'  Solar-C: {t_solarc:.1f} s ({t_solarc/t_dkist:.1f}x longer)')

In [ ]:
# Visualize the signal hierarchy
labels = ['Solar disk\ncontinuum', 'Corona\n(near AR)', 'Pol. signal\n(on corona)', 'Pol. signal\n(on disk)']
values = [1.0, 1e-6, 1e-4 * 1e-6, 1e-9]
colors = ['gold', 'orange', 'red', 'darkred']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(labels, values, color=colors, alpha=0.85, edgecolor='black')
ax.set_yscale('log')
ax.set_ylabel('Intensity relative to disk continuum')
ax.set_title('Coronal magnetometry is photon-starved (log scale)')
ax.grid(True, which='both', axis='y', alpha=0.3)
ax.axhline(5e-4, color='blue', linestyle='--',
           label='DKIST polarimetric accuracy floor (5e-4 of continuum)')
ax.legend(loc='upper right')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val,
            f' {val:.0e}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

## Part 5: Heat Load at Prime Focus / Prime Focus 열 부하

Total solar flux on DKIST M1:

$$
P_\odot = F_\odot \cdot A_{\text{M1}} \cdot \tau_{\text{atm}}
$$

At f/2 prime focus, the full solar disk image has diameter $\approx f_2 \cdot \theta_\odot \approx 75$ mm. The heat stop passes only the 5 arcmin science FOV, rejecting ~98% of the 13 kW load. The flux density at prime focus is **2.5 MW/m²**.

Heat stop는 5 arcmin 과학 FOV만 통과시키고 13 kW 중 98%를 제거한다. Prime focus 열유속 밀도는 **2.5 MW/m²**.

In [ ]:
def dkist_heat_load(D_m=4.0, f_ratio=2.0, atm_transmission=0.75,
                    fov_science_arcmin=5.0, solar_disk_arcmin=32.0):
    """Compute DKIST heat load and prime focus flux density."""
    A_M1 = np.pi * (D_m / 2.0)**2
    P_total = SOLAR_FLUX * A_M1 * atm_transmission

    f_length = f_ratio * D_m
    solar_disk_rad = solar_disk_arcmin * 60.0 * ARCSEC_TO_RAD
    disk_diameter_m = f_length * solar_disk_rad
    disk_area_m2 = np.pi * (disk_diameter_m / 2.0)**2

    flux_density = P_total / disk_area_m2

    # Fraction passed by heat stop (science FOV / solar disk, area ratio)
    passed_fraction = (fov_science_arcmin / solar_disk_arcmin)**2
    P_passed = P_total * passed_fraction
    P_rejected = P_total - P_passed

    return {
        'A_M1_m2': A_M1,
        'P_total_kW': P_total / 1e3,
        'disk_diameter_mm': disk_diameter_m * 1e3,
        'flux_density_MW_per_m2': flux_density / 1e6,
        'passed_fraction': passed_fraction,
        'P_passed_W': P_passed,
        'P_rejected_kW': P_rejected / 1e3,
    }


result = dkist_heat_load()
print('DKIST heat-load calculation at prime focus (f/2):')
print('-' * 55)
for k, v in result.items():
    print(f'  {k:<28s}: {v:>10.4g}')

print(f'\nPaper quotes: 13 kW total, 2.5 MW/m^2 flux density, ~98% rejected.')
print(f'Our calculation: {result["P_total_kW"]:.1f} kW, '
      f'{result["flux_density_MW_per_m2"]:.2f} MW/m^2, '
      f'{(1 - result["passed_fraction"]) * 100:.2f}% rejected.')

## Part 6: Polarimetric Photon Budget / 편광 광자 예산

To achieve polarimetric sensitivity $\sigma_{\text{pol}} = 5 \times 10^{-4}$ (photon-limited), we need:

$$
\sigma_{\text{pol}} = \frac{1}{\sqrt{N}} \;\Longrightarrow\; N = \frac{1}{\sigma_{\text{pol}}^2} = 4 \times 10^6 \text{ photons}
$$

per resolution element per modulation state.

$5 \times 10^{-4}$ 편광 정확도 달성에는 resolution element당 modulation state당 **400만 광자**가 필요하다.

In [ ]:
def photon_limited_polarimetric_accuracy(N_photons):
    """Shot-noise limit on polarimetric accuracy."""
    return 1.0 / np.sqrt(N_photons)


def required_photons_for_accuracy(sigma_target):
    """Number of photons required to reach target polarimetric accuracy."""
    return 1.0 / sigma_target**2


# DKIST targets and typical instrument sensitivities
targets = {
    'DKIST accuracy': 5e-4,
    'DKIST sensitivity floor (stated)': 1e-5,
    'VTF polarimetric sensitivity (13 s)': 5e-3,
    'Continuum stability req': 5e-4,
    'Ultimate (photon statistics)': 1e-6,
}

print(f"{'Target':<42s} {'sigma':>10s} {'N photons':>15s}")
print('-' * 70)
for name, sigma in targets.items():
    N = required_photons_for_accuracy(sigma)
    print(f"{name:<42s} {sigma:>10.1e} {N:>15.2e}")

# Verify: for 5e-4 accuracy, N = 4e6 photons
N_check = required_photons_for_accuracy(5e-4)
sigma_check = photon_limited_polarimetric_accuracy(N_check)
print(f'\nSelf-check: N = {N_check:.2e} -> sigma = {sigma_check:.2e}')

In [ ]:
# Plot photon-limited accuracy vs photon count
N_range = np.logspace(3, 10, 200)
sigma_range = photon_limited_polarimetric_accuracy(N_range)

fig, ax = plt.subplots(figsize=(10, 6))
ax.loglog(N_range, sigma_range, linewidth=2, color='navy', label='Photon-limited: 1/sqrt(N)')

# Mark DKIST targets
for name, sigma in [('DKIST accuracy (5e-4)', 5e-4),
                    ('VTF in 13 s (5e-3)', 5e-3),
                    ('DKIST sensitivity floor (1e-5)', 1e-5)]:
    N = required_photons_for_accuracy(sigma)
    ax.scatter([N], [sigma], s=100, zorder=5)
    ax.annotate(name, (N, sigma), fontsize=9,
                textcoords='offset points', xytext=(10, 10))

ax.set_xlabel('Photons per resolution element (per modulation state)')
ax.set_ylabel('Polarimetric accuracy sigma')
ax.set_title('Photon-limited polarimetric accuracy')
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('\nDKIST science case (Harrington et al. 2020a): a dual fiber-fed')
print('calibration spectrograph fits 3000+ Mueller-matrix variables.')
print('Photon statistics set the floor; system modeling determines the ceiling.')

## Part 7: DKIST vs. World Solar Telescopes / DKIST 대 세계 태양 망원경 비교

Comprehensive comparison visualizing DKIST's place among ground-based solar facilities.

지상 태양 관측 시설 사이에서 DKIST의 위치를 시각화.

In [ ]:
telescopes = {
    # name: (year_first_light, aperture_m, has_AO, is_off_axis, color)
    'McMath-Pierce':  (1962, 1.5,  False, False, 'gray'),
    'DST':            (1969, 0.76, True,  False, 'lightblue'),
    'VTT':            (1989, 0.7,  True,  False, 'lightblue'),
    'SST':            (2002, 1.0,  True,  False, 'steelblue'),
    'GREGOR':         (2012, 1.5,  True,  False, 'cornflowerblue'),
    'GST (BBSO)':     (2010, 1.6,  True,  True,  'royalblue'),
    'DKIST':          (2019, 4.0,  True,  True,  'crimson'),
    'EST (planned)':  (2029, 4.0,  True,  True,  'lightcoral'),
    'CGST (planned)': (2035, 8.0,  True,  True,  'darkred'),
}

fig, ax = plt.subplots(figsize=(12, 7))
for name, (year, D, has_ao, off_axis, color) in telescopes.items():
    size = (D * 60)**2
    marker = 'o' if off_axis else 's'
    ax.scatter(year, D, s=size, c=color, marker=marker, alpha=0.7,
               edgecolors='black', linewidths=1.5, zorder=3)
    ax.annotate(name, (year, D), fontsize=9,
                textcoords='offset points', xytext=(8, 8))

# Resolution milestones
for km, label in [(70, 'Pressure scale height'), (20, 'Photon mean-free path')]:
    D_limit = 1.22 * 1.6e-6 / (km * 1e3 / AU)
    ax.axhline(D_limit, linestyle='--', alpha=0.4)
    ax.text(2040, D_limit + 0.1, f'{label} @ 1.6 um',
            fontsize=9, ha='right', alpha=0.7)

ax.set_xlabel('Year of first light')
ax.set_ylabel('Aperture D [m]')
ax.set_title('Ground-based solar telescopes: aperture vs. era\n'
             '(Circles = off-axis design, Squares = classical)')
ax.grid(True, alpha=0.3)
ax.set_xlim(1955, 2045)
ax.set_ylim(0, 9)
plt.tight_layout()
plt.show()

print('\nDKIST opens the 4 m era. Note the doubling from GST (1.6 m) to DKIST (4 m) —')
print('the collecting-area jump is 6.25x, enabling coronal magnetometry for the first time.')

## Summary / 요약

| Concept / 개념 | DKIST Spec / DKIST 사양 | This Notebook / 본 노트북 |
|---|---|---|
| Diffraction limit / 회절 한계 | 0.026″ = 20 km @ 500 nm, 4 m | Part 1: Airy formula verified |
| IR Zeeman advantage / IR 제이만 이점 | $\lambda^2$ scaling, ~8× at 1.56 μm | Part 2: explicit comparison |
| AO Strehl / AO Strehl | S > 0.3 @ 500 nm, $r_0 > 7$ cm | Part 3: error-budget model |
| Coronal magnetometry / 코로나 자기장 | 75× Solar-C area, $10^{-9}$ signal | Part 4: integration-time scaling |
| Heat stop / 열 차단 | 13 kW, 2.5 MW/m², 98% rejected | Part 5: geometry reproduced |
| Polarimetry / 편광 측정 | $5 \times 10^{-4}$ accuracy | Part 6: photon budget = 4×10⁶ |
| Global context / 전체 맥락 | 4 m era begins | Part 7: telescope timeline |

### Key takeaway / 핵심 교훈

DKIST's specifications are not arbitrary — each arises from a specific physical constraint:
- **4 m aperture** ← resolve solar critical scales at IR wavelengths
- **Off-axis design** ← reduce scattered light for coronal work
- **$5 \times 10^{-4}$ polarimetry** ← measure weak-field Hanle diagnostics
- **2 kHz AO with 1,600 actuators** ← correct atmospheric turbulence over 4 m aperture
- **13 kW thermal design** ← manage solar heat without inducing local seeing

DKIST의 사양은 임의의 선택이 아니라 각각 특정 물리적 제약에서 수학적으로 도출된다.